# Crypto · arquitectura medallion, sin Microsoft Fabric

Reproduce el análisis de **Crypto_Medallion_Analytics** leyendo los CSV del propio repositorio desde GitHub. Se ejecuta de arriba abajo, sin instalar nada ni descargar archivos.

**Un aviso deliberado:** `crypto_historical_365days.csv` está vacío en el repositorio. La primera celda lo comprueba y lo dice en voz alta en lugar de disimularlo; el análisis sigue con la copia mensual y la anual, que sí tienen datos.

In [ ]:
import io, urllib.parse, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/Nicolenki7/Crypto_Medallion_Analytics/main/data/"

def leer(nombre):
    with urllib.request.urlopen(BASE + urllib.parse.quote(nombre)) as r:
        return pd.read_csv(io.BytesIO(r.read()))

# 1) el archivo que el proyecto documenta como vacio
with urllib.request.urlopen(BASE + "crypto_historical_365days.csv") as r:
    crudo = r.read()
try:
    historico = pd.read_csv(io.BytesIO(crudo))
    print("historico:", historico.shape)
except pd.errors.EmptyDataError as e:
    historico = None
    print(f"crypto_historical_365days.csv ({len(crudo)} bytes) no se puede leer: {e}")
    print("El dato de origen esta vacio. Es el problema que el informe del repo documenta.")

# 2) las tablas que si tienen datos
mensual = leer("crypto_monthly_summary - copia.csv")
anual = leer("crypto_yearly_performance - copia.csv")
print()
print("mensual:", mensual.shape, list(mensual.columns))
print("anual  :", anual.shape, list(anual.columns))

In [ ]:
mensual["month"] = pd.to_datetime(mensual["month"] + "-01")
ax = mensual.set_index("month")["avg_price"].plot(
    kind="line", marker="o", figsize=(9, 4), color="#2a9d8f")
ax.set_title("Precio medio mensual (USD)")
ax.set_xlabel("")
ax.set_ylabel("avg_price")
plt.tight_layout()
plt.show()

print(f"Meses analizados: {len(mensual)} | precio medio: {mensual['avg_price'].mean():,.2f} USD")
print(f"Volumen total: {mensual['total_volume'].sum():,.2e}")
mensual[["month", "avg_price", "avg_daily_return"]].tail()

In [ ]:
anual_ordenado = anual.sort_values("total_return", ascending=False)
top = anual_ordenado.head(10)

ax = top.set_index("coin_name")["total_return"].plot(
    kind="barh", figsize=(9, 4), color="#e76f51")
ax.invert_yaxis()
ax.set_title("Top 10 por retorno total del periodo (%)")
ax.set_xlabel("total_return (%)")
plt.tight_layout()
plt.show()

print(f"Monedas analizadas: {len(anual)}")
print(f"Mediana de retorno: {anual['total_return'].median():,.2f}%")
print(f"Mejor:  {top.iloc[0]['coin_name']} ({top.iloc[0]['total_return']:,.2f}%)")
print(f"Peor:   {anual_ordenado.iloc[-1]['coin_name']} ({anual_ordenado.iloc[-1]['total_return']:,.2f}%)")
top[["coin_name", "start_price", "end_price", "total_return"]]

## Por que esto es un caso de ingenieria y no un grafico

Las tres capas del proyecto (bronze, silver, gold) existen para que un fallo como *"el CSV de origen vino vacio"* no llegue al dashboard. Aqui se ve el orden correcto: validar, declarar el problema, y analizar solo lo que esta verificado.

Los numeros de estas celdas salen de los CSV del repositorio, no de una captura: si el archivo cambia, los resultados cambian con el.